# 07 - Capture SCiO scans over USB

This notebook connects to a Consumer Physics SCiO over USB, reads its metadata,
performs a **white reference** (calibration) and a **sample scan**, and stores
the raw encrypted blobs to disk. It also reads (read-only) the firmware file
headers, whose checksums let you match firmware blobs recovered from an old
phone (see notebook `08`).

**What works and what does not.** The device returns three blobs per scan
(dark, sample, gradient); each is an 8-byte plaintext header plus an
AES-block-sized encrypted body. Consumer Physics decoded these on their server,
which is now closed for this end-of-life device. So this notebook **captures and
stores** raw scans; turning them into a spectrum needs the device key, which is
the subject of notebook `08` (firmware key recovery).

**Safety.** Only read-only / capture commands are sent. Write and state-changing
commands (parameter set, LED, file download, reset, rename) are never issued
automatically; the transport refuses them unless you explicitly pass
`allow_write=True`.

Environment: run the `tp` conda env (or any env with `pyserial`, `numpy`,
`matplotlib`). Turn the SCiO on (long press -> blue) and plug in USB.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd() / "src"))
# also dev/: these notebooks use a few offline-research helpers (entropy, firmware triage)
sys.path.insert(1, str(Path.cwd() / "dev"))
from scio import usb, store, protocol

# Calibration staleness policy (you choose these; the app got them from the
# server). A threshold of 0 disables that check.
THRESHOLDS = {"time_ms": 24 * 3600 * 1000, "scans": 0, "temperature": 3.0}
FORCE_CALIBRATE = False

## 1. Find and connect

In [ ]:
ports = usb.find_scio_ports()
for p in ports:
    print(("* " if p["is_scio"] else "  ") + f'{p["device"]:8s} {p["vidpid"]}  {p["description"]}')
scio_port = next((p["device"] for p in ports if p["is_scio"]), None)
assert scio_port, "SCiO (VID:PID 0451:16AA) not found. Turn it on with a long press and replug USB."
print("Using", scio_port)

In [ ]:
dev = usb.ScioUSB(scio_port).open()
info = dev.read_device_info()
for k, v in info.items():
    print(f"  {k:18s} {v}")

## 2. Health: battery and temperature

In [ ]:
battery = dev.read_battery()
temp = dev.read_temperature()
print("battery:", battery)
print("temperature:", {k: round(v, 2) for k, v in temp.items()})

## 3. White reference (calibration)

The white reference is the *same* scan command taken with the SCiO sitting in
its cover / on its calibration surface. It is stored and reused across sessions;
the app recalibrates when it is too old, too many scans have passed, or the
temperature has drifted (mirrored by `store.calibration_status`).

In [ ]:
cal = store.load_latest_calibration(info["device_id"])
status = store.calibration_status(cal, temp["cmos_t"], THRESHOLDS)
print("calibration status:", status)

if FORCE_CALIBRATE or status != "NO_NEED":
    input("Place the SCiO in its cover / on the calibration surface, then press Enter...")
    tb = dev.read_temperature()
    wr = dev.white_reference(info["firmware_version"])
    ta = dev.read_temperature()
    wr_path = store.save_calibration(wr, info, tb, ta)
    print("saved white reference:", wr_path)
    cal = store.load_latest_calibration(info["device_id"])
else:
    print("Reusing existing white reference:", cal["path"])

## 4. Sample scan

In [ ]:
input("Aim the SCiO at your sample, then press Enter to scan...")
tb = dev.read_temperature()
scan = dev.sample_spectrum(info["firmware_version"])
ta = dev.read_temperature()
temps = {"before": {k: round(v, 2) for k, v in tb.items()},
         "after":  {k: round(v, 2) for k, v in ta.items()}}
for k, v in scan["blobs"].items():
    hdr = protocol.parse_blob_header(v)
    print(f'  {k:16s} {len(v):5d} B  type=0x{hdr["type"]:X} nonce=0x{hdr["nonce"]:08X} blocks={hdr["body_blocks"]:.0f}')
print("status word:", scan["status_word"])

scan_path = store.save_scan(scan["blobs"], info, temps, scan["status_word"],
                            calibration_file=cal["path"] if cal else None)
print("saved scan:", scan_path)

## 5. Read-only firmware file headers

These checksums (word 3 of each header) identify which per-device
image-to-spectrum tables and DSP firmware the SCiO holds. Save them next to the
device info; notebook `08` matches them against firmware blobs recovered from an
old phone.

In [ ]:
headers = dev.read_all_file_headers()      # ids 89-92 (firmware), 100-103 (tables)
file_list = dev.read_file_list()
record = {"device": info, "file_list": file_list,
          "file_headers": {str(k): v for k, v in headers.items()}}
dev_path = store.save_device_files(record)
for fid, h in headers.items():
    name = protocol.FIRMWARE_FILE_NAMES.get(fid, str(fid))
    print(f'  {name:18s} (id {fid})  checksum={h.get("checksum")}')
print("saved:", dev_path)

## 6. Look at the raw blobs (they are encrypted)

A quick view of the raw bytes: high entropy, no visible structure - this is the
encryption we need the device key to undo. Notebook `08` picks up from here.

In [ ]:
from scio_offline.decode import split_blob, entropy
fig, axes = plt.subplots(1, len(scan["blobs"]), figsize=(4 * len(scan["blobs"]), 3), squeeze=False)
for ax, (name, blob) in zip(axes[0], scan["blobs"].items()):
    b = split_blob(blob)
    ax.imshow(np.frombuffer(b.body, np.uint8).reshape(-1, 16), aspect="auto", cmap="viridis")
    ax.set_title(f"{name}  entropy {entropy(b.body):.2f} bit/B")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
dev.close()
print("Disconnected. Raw scan saved; run notebook 08 to work on decoding.")